# 01 Auditoria de qualidade dos dados

Antes de analisar, provar que o dado é o que aparenta. Este notebook existe para que cada decisão de limpeza tenha evidência registrada.

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

from src.data.load import build_panel, build_transitions, load_config
from src.data.clean import limpar
from src.data.validate import validar_painel, validar_transicoes, resumo_carga

cfg = load_config('../config.yaml')
RAW = '../' + cfg['paths']['raw']

In [2]:
painel = build_panel(RAW)
print(painel.shape, '|', painel.ra.nunique(), 'alunos')
painel.head()

(3030, 37) | 1661 alunos


,ra,ano,fase_num,fase_raw,turma,idade,genero,ano_ingresso,instituicao,inde,...,fase_ideal_num,fase_ideal_raw,defasagem,rec_psicologia,destaque_ieg,destaque_ida,destaque_ipv,escola,n_indic_faltantes,sem_avaliacao
0,RA-1,2022,7.0,7,A,19.0,F,2016,Escola Pública,5.783,...,8.0,Fase 8 (Universitários),-1.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,NaN,0,0
1,RA-1,2023,8.0,FASE 8,8E,NaN,F,2016,Privada *Parcerias com Bolsa 100%,NaN,...,8.0,Fase 8 (Universitários),0.0,NaN,NaN,NaN,NaN,NaN,6,1
2,RA-1,2024,8.0,8E,8E,21.0,F,2021,Privada *Parcerias com Bolsa 100%,NaN,...,8.0,Fase 8 (Universitários),0.0,NaN,NaN,NaN,NaN,Universidade Santo Amaro (UNISA),5,1
3,RA-10,2022,7.0,7,A,18.0,F,2021,Escola Pública,5.784,...,8.0,Fase 8 (Universitários),-1.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,NaN,0,0
4,RA-100,2022,4.0,4,A,13.0,F,2019,Rede Decisão,7.618,...,3.0,Fase 3 (7º e 8º ano),1.0,Não indicado,Destaque: A sua boa entrega das lições de casa.,Destaque: As suas boas notas na Passos Mágicos.,Melhorar: Integrar-se mais aos Princípios Pass...,NaN,0,0


O dicionário não descreve essa base

In [3]:
# Cobertura por coluna e ano — o que existe de verdade em cada aba
cob = painel.groupby('ano').count().T
cob[cob.min(axis=1) == 0]  # colunas vazias em pelo menos um ano

ano,2022,2023,2024
cg,860,0,0
cf,860,0,0
ct,860,0,0
ipp,0,938,1054
indicado_bolsa,860,0,0
atingiu_pv,860,0,0
rec_psicologia,860,0,0
destaque_ieg,860,0,0
destaque_ida,860,0,0
destaque_ipv,860,0,0


### Colunas-zumbi

`Atingiu PV`, `Rec Psicologia`, `Indicado` e `CG` só têm dado em 2022. `IPP` só em 2023–24. `Escola` só em 2024.

Isso mata o alvo mais natural do briefing (ponto de virada) e assimetriza o conjunto de features entre as janelas de treino e teste.

In [4]:
IND = ['iaa','ieg','ips','ipp','ida','ipv','ian','inde']
painel.groupby('ano')[IND].describe().T.round(2)

ano           2022    2023     2024
iaa  count  860.00  951.00  1054.00
     mean     8.27    6.90     8.54
     std      2.06    3.59     1.49
     min      0.00    0.00     0.00
     25%      7.90    6.70     8.00
...            ...     ...      ...
inde min      3.03    3.75     3.79
     25%      6.49    6.72     6.77
     50%      7.20    7.41     7.54
     75%      7.75    8.00     8.14
     max      9.44    9.37     9.53

[64 rows x 3 columns]

## H10 os indicadores mudaram de método, ou os alunos mudaram?

Oscilação em V (cair e voltar) não é comportamento de população. O teste roda na corte fechada só alunos presentes nos três anos para eliminar composição como explicação.

In [5]:
from src.analysis.eda_tecnica import coorte_fechada, testar_drift

cols = ['iaa','ieg','ips','ida','ipv','defasagem']
co = coorte_fechada(painel, cols)
print(f'coorte fechada: {len(co)} alunos')
testar_drift(painel[painel.ra.isin(co)], cols)

coorte fechada: 434 alunos


,indicador,n,med_2022,med_2023,med_2024,amplitude,p_friedman,veredito
0,iaa,434,9.00,8.50,8.75,0.50,2.276815e-12,DRIFT (padrao em V)
1,ieg,434,8.80,9.00,8.38,0.62,2.434109e-12,DRIFT (padrao em V)
2,ips,434,7.50,5.00,7.50,2.50,1.827054e-19,DRIFT (padrao em V)
3,ida,434,7.05,6.85,6.50,0.55,1.973680e-03,MUDANCA REAL (monotonica)
4,ipv,434,7.50,8.10,7.44,0.66,3.948336e-34,DRIFT (padrao em V)
5,defasagem,434,-1.00,-1.00,0.00,1.00,5.634964e-44,MUDANCA REAL (monotonica)


Veredito. O drift é específico do **IPS** (mediana 7,50 → 5,00 → 7,51) e moderado no **IPV**. IAA, IEG e IDA estão estáveis.

O controle negativo: `defasagem` se move com amplitude comparável, mas de forma monotônica é melhora real.

Consequência: IPS e IPV entram no modelo em percentil intra-ano, e a efetividade do programa se argumenta por defasagem, nunca por INDE.

In [6]:
# Granularidade numérica: prova documental da mudança de cálculo
for c in ['ieg','ipv']:
    for a in [2022, 2023, 2024]:
        s = painel.loc[painel.ano == a, c].dropna()
        casas = s.map(lambda x: len(str(float(x)).split('.')[1].rstrip('0'))).max()
        print(f'{c} {a}: {s.nunique():4d} valores distintos, até {casas} casas decimais')

ieg 2022:   75 valores distintos, até 1 casas decimais
ieg 2023:   54 valores distintos, até 1 casas decimais
ieg 2024:  741 valores distintos, até 16 casas decimais
ipv 2022:  170 valores distintos, até 3 casas decimais
ipv 2023:  394 valores distintos, até 9 casas decimais
ipv 2024:  530 valores distintos, até 9 casas decimais


 H11 zeros que não são notas

`IAA = 0` em 190 registros de 2023, quase 19% do ano.

In [7]:
for c in ['iaa','ieg','ida']:
    s = painel[c].dropna()
    prox = np.unique(s[s > 0])[:4]
    print(f'{c}: {int((s==0).sum()):4d} zeros | menores valores > 0: {np.round(prox,3)}')

iaa:  249 zeros | menores valores > 0: [1.7   3.498 3.5   4.   ]
ieg:  110 zeros | menores valores > 0: [2.    2.524 2.7   2.751]
ida:   23 zeros | menores valores > 0: [0.5 0.7 0.9 1. ]


**Veredito por indicador a regra uniforme seria erro.**

| Indicador | Zeros | Decisão | Razão |
|---|---|---|---|
| IAA | 249 | → `NaN` + flag | Gap de 0 até 1,7–3,5; não-persistente entre anos |
| IEG | 110 | manter | Gap menor, e engajamento nulo é estado real |
| IDA | 23 | manter | Continuum 0 / 0,5 / 0,7 / 0,9 |

In [8]:
painel_limpo, log = limpar(painel)
log

,coluna,regra,n_afetado,detalhe
0,iaa,zero -> NaN (sentinela),249,flag _nao_respondeu criada
1,ieg,zero MANTIDO (medida real),110,distribuicao continua
2,ida,zero MANTIDO (medida real),23,distribuicao continua
3,iaa,truncado em 10.0,122,excesso max 0.002
4,ipv,truncado em 10.0,26,excesso max 0.010
5,idade,recuperada por aritmetica no painel,362,restam 37 sem referencia
6,instituicao,-> binario rede_publica,3013,17 nao classificados
7,turma,-> tamanho_turma (por ano+fase+turma),3030,120 niveis descartados


In [9]:
# O IAA depois da limpeza: a oscilação era artefato dos zeros
pd.DataFrame({'antes': painel.groupby('ano').iaa.mean(),
              'depois': painel_limpo.groupby('ano').iaa.mean()}).round(2)

,antes,depois
ano,,
2022,8.27,8.67
2023,6.90,8.63
2024,8.54,8.71


In [10]:
avisos = validar_painel(painel_limpo, cfg, estrito=False)
print('contratos:', avisos or 'todos OK')

trans = build_transitions(painel_limpo)
print('transições:', validar_transicoes(trans, estrito=False) or 'todos OK')
resumo_carga(painel_limpo, trans)[1]

contratos: todos OK
transições: todos OK


,transicoes,eventos,base_rate
janela,,,
2022->2023,600,104,0.173
2023->2024,765,132,0.173


## Fluxo do painel

Base do viés de sobrevivência que acompanha toda análise longitudinal daqui em diante.

In [11]:
from src.analysis.eda_tecnica import fluxo_painel
fluxo_painel(painel_limpo)

,transicao,base_inicial,permanecem,saem,taxa_saida,entram
0,2022->2023,860,600,260,0.302,414
1,2023->2024,1014,765,249,0.246,391


25–30% saem por ano, e **apenas 4 alunos** retornaram após ausência em todo o painel. Na prática, a saída é definitiva.

In [12]:
from src.data.load import salvar
salvar(painel_limpo, '../data/interim/painel_limpo')
log.to_csv('../data/interim/log_limpeza.csv', index=False, encoding='utf-8-sig')
print('exportado')

exportado
